# TUGAS RECIPE BAB 2.4 - AI Cookbook for Earth Sciences
Nama: Aulia Zezarya
NIM: H061251063
Kelas : B
Mata Kuliah: AI Cookbook for Earth Sciences


RECIPE 1: R-IO-001 - AUDIT QA/QC STATION CSV (DARI BUKU HAL 29)

ID: R-IO-001
Prompt ID: P-IO-001
Status: Not yet executed -> Menjadi Executed setelah run python code/recipes/io/r-io-001.py

1. PROBLEM: Menilai kelayakan tabel station CSV tanpa mengubah raw observation. Satu outcome utama = report QA/QC Markdown.

2. MINIMAL THEORY: Timestamp ganda & missing mengubah agregasi bulanan. Audit harus read-only, no imputation/deletion. Timezone harus eksplisit UTC.

3. INPUT/CONSTRAINT:
Input: [CSV_PATH]=data/samples/station-example.csv, [TIME_COLUMN]=time, [UNITS]=mm/C UTC, [REPORT_PATH]=results/io/station-summary.md
Constraint: Jangan imputasi, jangan hapus baris, no-network, output Markdown saja.
Output: results/io/station-summary.md + results/run-records/r-io-001.json

4. PROMPT REUSABLE P-IO-001:
Role: Data steward GIS. Task: Baca [CSV_PATH] tanpa mengubahnya. Identifikasi schema (columns, types), time range min/max, missing count per kolom, duplicate timestamp di [TIME_COLUMN]. Output Markdown di [REPORT_PATH]. Verification: row count preserved.

5. IMPLEMENTATION: code/recipes/io/r-io-001.py
Menggunakan pandas read_csv, parse_dates [TIME_COLUMN], isna().sum(), duplicated().

6. EXPECTED RESULT: Shape 5 rows 4 cols, 1 missing di temperature_C, 1 duplicate timestamp (sesuai contoh buku Hal 28).

7. FAILURE MODES: Jika [CSV_PATH] tidak ada -> fail exit 1 path not found. Jika [TIME_COLUMN] tidak ada -> fail schema invalid. Jika [UNITS] tidak eksplisit -> minta clarification.

8. STATUS EVIDENCE: Not yet executed sampai ada bukti command, exit code 0, row count, dan file report. WARNING Hal 30: Jangan klaim Executed sebelum run.

RECIPE 2: R-GEO-001 - NORMALISASI GEOJSON (DARI BUKU HAL 29)

ID: R-GEO-001
Status: Not yet executed

1. PROBLEM: Ubah [GEOJSON_PATH] menjadi normalized.geojson + inventory.csv + summary.json tanpa hilang Feature ID, geometry, properties, null geometry.
2. MINIMAL THEORY: Preserve Feature ID, preserve coordinate order lon/lat, CRS/axis/vertical harus dinyatakan eksplisit. No reprojection unless requested.
3. INPUT/CONSTRAINT: [GEOJSON_PATH], Constraint: No-network, no-reproject. Jika CRS tidak ada -> stop minta clarification jangan tebak. Output: results/geo/normalized.geojson, inventory.csv, summary.json
4. PROMPT: Normalisasi GeoJSON preserve semua ID dan geometry.
5. FAILURE: Duplicate Feature ID -> flag. CRS tidak diketahui -> fail.
6. STATUS: Not yet executed.

RECIPE 3: R-THERM-001 - DIFUSI PANAS 1D (DARI BUKU HAL 29)

ID: R-THERM-001
Status: Not yet executed

1. PROBLEM: Simulasi difusi panas 1D vertikal.
2. MINIMAL THEORY: Governing equation dT/dt = ALPHA * d2T/dz2. Stability condition DT <= DZ^2 / (2*ALPHA). Depth positive down. Unit degC, m, s.
3. INPUT: [INITIAL_PROFILE], [ALPHA]=1e-6 m2/s, [BOUNDARIES]=top 20C bottom Neumann 0, [DZ]=0.1 m, [DT]=100 s, [T_END]. Config: configurations/r-therm-001.json
4. OUTPUT: results/therm/profile.csv + figures/therm/profile.png + results/run-records/r-therm-001.json
5. FAILURE: Jika DT tidak memenuhi stability -> fail minta kurangi DT.
6. STATUS: Not yet executed.

RECIPE 4: R-IO-002 OWN - RECIPE SENDIRI - AUDIT HUJAN MAKASSAR

ID: R-IO-002 (OWN - BUATAN SENDIRI) | Related: R-IO-001
Prompt ID: P-IO-002 (Buatan Sendiri)
Status: Not yet executed - RECIPE SENDIRI

1. PROBLEM: Audit kelengkapan data hujan harian 3 stasiun Makassar (Paotere, Biringkanaya, Maros) periode 2024-2025 sebelum agregasi total bulanan dan indeks kekeringan. Keputusan: layak agregasi atau perlu review lapangan. Satu outcome = report QA/QC per stasiun.

2. MINIMAL THEORY: Missing data terjadi karena tipping bucket rusak saat hujan ekstrem >100mm. Duplicate karena upload logger ganda. Agregasi bulanan bias jika missing >10%. Outlier <0 mm atau >500 mm harus flag. Timezone data logger WITA UTC+8 harus convert ke UTC untuk analisis. Constraint read-only.

3. INPUT/CONSTRAINT:
Input: [CSV_PATH]=data/samples/makassar-rainfall-2024.csv, Schema: station_id, time (WITA), rainfall_mm (mm), unit mm, timezone Asia/Makassar, Output: results/io/makassar-audit.md + results/run-records/r-io-002.json
Constraint: Read-only, no imputation, no deletion, no network, output Markdown.
4. PROMPT P-IO-002 (Buatan Sendiri):
Role: Data steward GIS + Scientific Python developer.
Task: Baca [CSV_PATH] tanpa mengubahnya. Convert time WITA->UTC. Untuk setiap station_id hitung total rows, missing rainfall_mm count, duplicate time per station, outlier flag rainfall_mm <0 atau >500, time range min/max. Output Markdown di results/io/makassar-audit.md per station. Verification: row count preserved, groupby station_id.
Context: Skripsi iklim Makassar 2024-2025, aggregasi bulanan sensitif terhadap missing.

5. IMPLEMENTATION: code/recipes/io/r-io-002.py
Logic: pandas read_csv parse_dates, groupby station_id, isna().sum(), duplicated(subset=['time']), outlier filter.

6. CHECKS: Missing per station, duplicate list, outlier list, time min/max.

7. FAILURE: File tidak ada -> path not found. Kolom rainfall_mm tidak ada -> schema invalid.

8. STATUS EVIDENCE: Not yet executed sampai ada bukti run. Ini recipe sendiri memenuhi syarat Hal 27 buku.